MOUNT DRIVE TO COLAB

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


LOAD THE DATA

In [31]:
import pandas as pd
data = pd.read_csv("/content/drive/MyDrive/warehouse_messy_data.csv")

INSPECT THE DATASET

In [32]:
print(data.info())
print(data.isnull().sum())
print(data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Product ID      1000 non-null   int64  
 1   Product Name    1000 non-null   object 
 2   Category        1000 non-null   object 
 3   Warehouse       1000 non-null   object 
 4   Location        1000 non-null   object 
 5   Quantity        842 non-null    object 
 6   Price           793 non-null    float64
 7   Supplier        1000 non-null   object 
 8   Status          1000 non-null   object 
 9   Last Restocked  800 non-null    object 
dtypes: float64(1), int64(1), object(8)
memory usage: 78.3+ KB
None
Product ID          0
Product Name        0
Category            0
Warehouse           0
Location            0
Quantity          158
Price             207
Supplier            0
Status              0
Last Restocked    200
dtype: int64
   Product ID Product Name     Category    Warehouse 

ISSUES FOUND
| Column         | Issue                                                   |
| -------------- | ------------------------------------------------------- |
| Quantity       | 158 missing values                                      |
| Price          | 207 missing values                                      |
| Last Restocked | 200 missing values                                      |
| Product Name   | Extra spaces                                            |
| Quantity       | Mixed values (e.g., `"two hundred"` and numeric values) |
| Product ID     | Many repeated IDs (potential duplicates)                |
| Last Restocked | Stored as text instead of date                          |


CLEAN PRODUCT NAMES
(REMOVE LEADING AND TRAILING SPACES):

In [34]:
data["Product Name"] = data["Product Name"].str.strip()

CLEAN QUANTITY COLUMN

In [35]:
data["Quantity"] = data["Quantity"].replace({
    "two hundred": 200
})

data["Quantity"] = pd.to_numeric(data["Quantity"], errors="coerce")

CHECK

In [36]:
print(data["Quantity"].unique())

[300. 200. 100.  50.  nan 150.]


HANDLE MISSING QUANTITY VALUES

In [37]:
data["Quantity"] = data["Quantity"].fillna(data["Quantity"].median())
print(data["Quantity"].isnull().sum())

0


HANDLE MISSING PRICES

In [38]:
data["Price"] = data["Price"].fillna(data["Price"].median())
print(data["Price"].isnull().sum())

0


CONVERT DATE COLUMN

In [39]:
data["Last Restocked"] = pd.to_datetime(
    data["Last Restocked"],
    format="%d/%m/%Y",
    errors="coerce"
)
data

,Product ID,Product Name,Category,Warehouse,Location,Quantity,Price,Supplier,Status,Last Restocked
0,1102,gadget y,ELECTRONICS,Warehouse 2,Aisle 1,300.0,9.99,Supplier C,In Stock,NaT
1,1435,gadget y,ELECTRONICS,Warehouse 2,Aisle 4,200.0,19.99,Supplier C,Out of Stock,NaT
2,1860,widget a,CLOTHING,Warehouse 2,Aisle 3,100.0,19.99,Supplier B,In Stock,2022-12-20
3,1270,gadget z,TOYS,Warehouse 2,Aisle 4,50.0,49.99,Supplier B,In Stock,2022-12-20
4,1106,widget a,FURNITURE,Warehouse 3,Aisle 3,200.0,9.99,Supplier D,Out of Stock,2023-04-25
...,...,...,...,...,...,...,...,...,...,...
995,1009,widget b,FURNITURE,Warehouse 2,Aisle 2,100.0,29.99,Supplier C,In Stock,2023-01-15
996,1823,gadget y,ELECTRONICS,Warehouse 2,Aisle 3,300.0,19.99,Supplier B,In Stock,2022-12-20
997,1797,gadget z,TOYS,Warehouse 3,Aisle 5,150.0,9.99,Supplier C,Low Stock,2023-03-05
998,1241,widget c,FURNITURE,Warehouse 2,Aisle 2,100.0,49.99,Supplier C,Low Stock,2022-12-20


FILL MISING DATES

In [40]:
data["Last Restocked"] = data["Last Restocked"].fillna(
    data["Last Restocked"].mode()[0]
)
data

,Product ID,Product Name,Category,Warehouse,Location,Quantity,Price,Supplier,Status,Last Restocked
0,1102,gadget y,ELECTRONICS,Warehouse 2,Aisle 1,300.0,9.99,Supplier C,In Stock,2022-12-20
1,1435,gadget y,ELECTRONICS,Warehouse 2,Aisle 4,200.0,19.99,Supplier C,Out of Stock,2022-12-20
2,1860,widget a,CLOTHING,Warehouse 2,Aisle 3,100.0,19.99,Supplier B,In Stock,2022-12-20
3,1270,gadget z,TOYS,Warehouse 2,Aisle 4,50.0,49.99,Supplier B,In Stock,2022-12-20
4,1106,widget a,FURNITURE,Warehouse 3,Aisle 3,200.0,9.99,Supplier D,Out of Stock,2023-04-25
...,...,...,...,...,...,...,...,...,...,...
995,1009,widget b,FURNITURE,Warehouse 2,Aisle 2,100.0,29.99,Supplier C,In Stock,2023-01-15
996,1823,gadget y,ELECTRONICS,Warehouse 2,Aisle 3,300.0,19.99,Supplier B,In Stock,2022-12-20
997,1797,gadget z,TOYS,Warehouse 3,Aisle 5,150.0,9.99,Supplier C,Low Stock,2023-03-05
998,1241,widget c,FURNITURE,Warehouse 2,Aisle 2,100.0,49.99,Supplier C,Low Stock,2022-12-20


CHECK DUPLICATE PRODUCT IDs

In [41]:
print(data["Product ID"].duplicated().sum())

385


Since a warehouse can stock the same product multiple times, first investigate before deleting rows.

In [42]:
data[data["Product ID"].duplicated(keep=False)]

,Product ID,Product Name,Category,Warehouse,Location,Quantity,Price,Supplier,Status,Last Restocked
0,1102,gadget y,ELECTRONICS,Warehouse 2,Aisle 1,300.0,9.99,Supplier C,In Stock,2022-12-20
2,1860,widget a,CLOTHING,Warehouse 2,Aisle 3,100.0,19.99,Supplier B,In Stock,2022-12-20
3,1270,gadget z,TOYS,Warehouse 2,Aisle 4,50.0,49.99,Supplier B,In Stock,2022-12-20
7,1020,widget c,CLOTHING,Warehouse 1,Aisle 5,200.0,9.99,Supplier D,Out of Stock,2022-12-20
9,1121,widget b,TOYS,Warehouse 1,Aisle 2,50.0,19.99,Supplier C,Out of Stock,2022-12-20
...,...,...,...,...,...,...,...,...,...,...
992,1814,gadget y,ELECTRONICS,Warehouse 3,Aisle 1,150.0,19.99,Supplier B,In Stock,2022-12-20
993,1920,gadget y,FURNITURE,Warehouse 1,Aisle 5,100.0,29.99,Supplier A,Out of Stock,2023-01-15
996,1823,gadget y,ELECTRONICS,Warehouse 2,Aisle 3,300.0,19.99,Supplier B,In Stock,2022-12-20
997,1797,gadget z,TOYS,Warehouse 3,Aisle 5,150.0,9.99,Supplier C,Low Stock,2023-03-05


STANDARDIZE TEXT COLUMNS                 
Example:

ELECTRONICS → Electronics,
FURNITURE → Furniture

In [43]:
data["Category"] = data["Category"].str.title()
data["Status"] = data["Status"].str.title()
data

,Product ID,Product Name,Category,Warehouse,Location,Quantity,Price,Supplier,Status,Last Restocked
0,1102,gadget y,Electronics,Warehouse 2,Aisle 1,300.0,9.99,Supplier C,In Stock,2022-12-20
1,1435,gadget y,Electronics,Warehouse 2,Aisle 4,200.0,19.99,Supplier C,Out Of Stock,2022-12-20
2,1860,widget a,Clothing,Warehouse 2,Aisle 3,100.0,19.99,Supplier B,In Stock,2022-12-20
3,1270,gadget z,Toys,Warehouse 2,Aisle 4,50.0,49.99,Supplier B,In Stock,2022-12-20
4,1106,widget a,Furniture,Warehouse 3,Aisle 3,200.0,9.99,Supplier D,Out Of Stock,2023-04-25
...,...,...,...,...,...,...,...,...,...,...
995,1009,widget b,Furniture,Warehouse 2,Aisle 2,100.0,29.99,Supplier C,In Stock,2023-01-15
996,1823,gadget y,Electronics,Warehouse 2,Aisle 3,300.0,19.99,Supplier B,In Stock,2022-12-20
997,1797,gadget z,Toys,Warehouse 3,Aisle 5,150.0,9.99,Supplier C,Low Stock,2023-03-05
998,1241,widget c,Furniture,Warehouse 2,Aisle 2,100.0,49.99,Supplier C,Low Stock,2022-12-20


VERIFY DATA TYPES

In [44]:
print(data.dtypes)

Product ID                 int64
Product Name              object
Category                  object
Warehouse                 object
Location                  object
Quantity                 float64
Price                    float64
Supplier                  object
Status                    object
Last Restocked    datetime64[ns]
dtype: object


SAVED CLEANED DATA

In [46]:
data.to_csv("/content/drive/MyDrive/warehouse_messy_data.csv", index=False)